# SBP calibration

Simulation by default. Replace example coordinates and configure motor limits before real use. [Hardware details](README.md#calibration-and-data-acquisition).

In [ ]:
from pathlib import Path
from src.calibration import Calibration, capture_endpoint, make_calibration
from src.acquisition import MeasurementConfig, build_motor, create_demo_config
from src.motor import jog_background, show_emergency_stop, list_serial_ports

SIMULATE = True
CONFIG_FILE = (create_demo_config("outputs/calibration") if SIMULATE
               else Path("config/measurement_config.json"))
config = MeasurementConfig.from_file(CONFIG_FILE)

## 1. Thread endpoints

XYZ values are examples. `XYZ_UNIT_M` sets their unit; margins are in metres.

In [ ]:
XYZ_UNIT_M = 1.0
SUB_THREADS = [
    {"index": 0, "name": "T0", "start_xyz": [-0.1, 0, 0], "end_xyz": [0.1, 0, 0]},
    {"index": 1, "name": "T1", "start_xyz": [-0.1, 0, 0.02], "end_xyz": [0.1, 0, 0.02],
     "margin_start_m": 0.01, "margin_end_m": 0.01},
    {"index": 2, "name": "T2", "start_xyz": [-0.1, 0, 0.04], "end_xyz": [0.1, 0, 0.04]},
    {"index": 3, "name": "T3", "start_xyz": [-0.1, 0, 0.06], "end_xyz": [0.1, 0, 0.06],
     "margin_start_m": 0.01, "margin_end_m": 0.01},
]

## 2. Motor and limits

Home is established manually. After emergency STOP, re-establish position before moving.

In [ ]:
# list_serial_ports()
motor = build_motor(config, simulate=SIMULATE)
motor.set_limits(config.stepper.min_position_steps, config.stepper.max_position_steps)
print(motor.limits_description())
print("Current position:", motor.get_position(), "steps")
stop_button = show_emergency_stop(motor)

### Position reference

Declare a position only while the bead is physically at that reference.

In [ ]:
# For a real motor, declare a known position only while the bead is at that reference:
# motor.set_position(known_steps)
# motor.set_home()
print("Stored home:", motor.home_position)

## 3. Capture endpoints

Simulation captures examples. For real calibration, capture start/end for every thread.

In [ ]:
captured = {}
if SIMULATE:
    example = Calibration.from_calibration_file("config/bead_pull_calibration.example.json")
    for sub in example.sub_threads:
        motor.move_to_position(sub.start_steps)
        capture_endpoint(motor, captured, sub.index, "start")
        motor.move_to_position(sub.end_steps)
        capture_endpoint(motor, captured, sub.index, "end")

In [ ]:
THREAD_INDEX = 0
# move = jog_background(motor, 200)
# move.join()  # wait before starting another movement
# capture_endpoint(motor, captured, THREAD_INDEX, "start")
# Jog to the other endpoint, then:
# capture_endpoint(motor, captured, THREAD_INDEX, "end")
captured

## 4. Check and save

Check endpoint positions and each thread’s steps-per-metre scale.

In [ ]:
calibration = make_calibration(SUB_THREADS, captured, coordinate_unit_m=XYZ_UNIT_M)
print(calibration.summary())
calibration.save(config.calibration_file)
reloaded = Calibration.from_calibration_file(config.calibration_file)
print("Saved:", config.calibration_file)

## 5. Disconnect

In [ ]:
motor.shutdown()